In [1]:
pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import chromadb
from chromadb.config import Settings
import re
from typing import Dict, List, Tuple
from typing import Optional
import logging
from datetime import datetime
from tabulate import tabulate
import anthropic
import json
import time


# Library to suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [39]:
logging.basicConfig(level=logging.INFO,
                   format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('anthropic').setLevel(logging.WARNING)

In [4]:
def load_env_from_json(file_path):
    """Loads security keys from a JSON file and sets them as environment variables."""
    try:
        with open(file_path, 'r') as file:
            secrets = json.load(file)
            for key, value in secrets.items():
                os.environ[key] = value
                print(f"Loaded {key} into environment variables")  # Optional, for debugging
    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except json.JSONDecodeError:
        print(f"Error: Could not parse {file_path}. Ensure it is valid JSON.")

In [5]:
load_env_from_json('secrets.json')

Loaded ANTHROPIC_API_KEY into environment variables


## Load Data and Parser Agent

In [6]:
class CarDataProcessor:
    def __init__(self, db_path: str = "./chromadb"):
        """Initialize the CarDataProcessor with ChromaDB connection."""
        self.client = chromadb.PersistentClient(path=db_path)
        
        # Create collections for each section if they don't exist
        self.specs_collection = self.client.get_or_create_collection(
            name="car_specifications",
            metadata={"description": "Basic car specifications"}
        )
        self.features_collection = self.client.get_or_create_collection(
            name="car_features",
            metadata={"description": "Car features and availability"}
        )
        self.safety_collection = self.client.get_or_create_collection(
            name="car_safety",
            metadata={"description": "Car safety ratings"}
        )
    
    def parse_filename(self, filename: str) -> Tuple[str, str, str]:
        """Extract year, make, and model from filename."""
        base = os.path.splitext(filename)[0]
        parts = base.split('_')
        if len(parts) != 3:
            raise ValueError(f"Invalid filename format: {filename}")
        return parts[0], parts[1], parts[2]

    def convert_to_dataframe(self, lines: List[str]) -> pd.DataFrame:
        """Convert a list of CSV lines into a pandas DataFrame."""
        if not lines:
            return pd.DataFrame()

        # Get headers from first line and clean them
        headers = []
        first_line = lines[0].split(',')
        for col in first_line:
            col = col.strip()
            # Remove any BOM or special characters
            col = col.replace('\ufeff', '').strip()
            if col:
                headers.append(col)
        
        # Process data rows
        data = []
        for line in lines[1:]:
            if line.strip():  # Skip empty lines
                values = [val.strip() for val in line.split(',')]
                # Ensure we have enough values to match headers
                while len(values) < len(headers):
                    values.append('')
                data.append(values[:len(headers)])
        
        return pd.DataFrame(data, columns=headers)
    
    def process_csv_sections(self, file_path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """Process CSV file line by line and split into three sections."""
        sections = []
        current_section = []
        started = False  # Flag to indicate if we've started collecting data
        
        with open(file_path, 'r', encoding='utf-8-sig') as file:  # Handle UTF-8 BOM
            for line in file:
                line = line.strip()
                if line.startswith('####'):
                    if current_section:  # If we have content in current section
                        sections.append(current_section)
                    current_section = []
                    started = True  # Start collecting after seeing ####
                elif started and line:  # Only add non-empty lines after seeing ####
                    current_section.append(line)
        
        # Don't forget the last section
        if current_section:
            sections.append(current_section)
        
        # Convert sections to DataFrames
        specs_df = pd.DataFrame()
        features_df = pd.DataFrame()
        safety_df = pd.DataFrame()
        
        if len(sections) >= 1:
            specs_df = self.convert_to_dataframe(sections[0])
        if len(sections) >= 2:
            features_df = self.convert_to_dataframe(sections[1])
        if len(sections) >= 3:
            safety_df = self.convert_to_dataframe(sections[2])
        
        return specs_df, features_df, safety_df
    
    def create_document_id(self, year: str, make: str, model: str, section: str, idx: int) -> str:
        """Create a unique document ID for ChromaDB."""
        return f"{year}_{make}_{model}_{section}_{idx}"
    
    def process_single_file(self, file_path: str) -> None:
        """Process a single CSV file and upload to ChromaDB."""
        try:
            # Extract car info from filename
            year, make, model = self.parse_filename(os.path.basename(file_path))
            logger.info(f"Processing {year} {make} {model}")
            
            # Process CSV sections
            specs_df, features_df, safety_df = self.process_csv_sections(file_path)
            
            # Process specifications
            if not specs_df.empty:
                for idx, row in specs_df.iterrows():
                    doc_id = self.create_document_id(year, make, model, "specs", idx)
                    self.specs_collection.add(
                        documents=[f"{row['Spec']}: {row['Value']}"],
                        metadatas=[{
                            "year": year,
                            "make": make,
                            "model": model,
                            "spec": row['Spec'],
                            "value": str(row['Value'])
                        }],
                        ids=[doc_id]
                    )
            
            # Process features
            if not features_df.empty:
                for idx, row in features_df.iterrows():
                    doc_id = self.create_document_id(year, make, model, "features", idx)
                    self.features_collection.add(
                        documents=[f"{row['features']}: {row['Availability']} (MSRP: ${row['MSRP']})"],
                        metadatas=[{
                            "year": year,
                            "make": make,
                            "model": model,
                            "feature": row['features'],
                            "availability": row['Availability'],
                            "msrp": str(row['MSRP'])
                        }],
                        ids=[doc_id]
                    )
            
            # Process safety ratings
            if not safety_df.empty:
                for idx, row in safety_df.iterrows():
                    doc_id = self.create_document_id(year, make, model, "safety", idx)
                    self.safety_collection.add(
                        documents=[f"{row['safety']}: {row['Rating']}"],
                        metadatas=[{
                            "year": year,
                            "make": make,
                            "model": model,
                            "safety_aspect": row['safety'],
                            "rating": str(row['Rating'])
                        }],
                        ids=[doc_id]
                    )
            
            logger.info(f"Successfully processed {year} {make} {model}")
            
        except Exception as e:
            logger.error(f"Error processing file {file_path}: {str(e)}")
            raise
    
    def process_directory(self, directory_path: str) -> None:
        """Process all CSV files in a directory."""
        processed_files = 0
        failed_files = 0
        
        for filename in os.listdir(directory_path):
            if filename.endswith('.csv'):
                file_path = os.path.join(directory_path, filename)
                try:
                    self.process_single_file(file_path)
                    processed_files += 1
                except Exception as e:
                    failed_files += 1
                    logger.error(f"Failed to process {filename}: {str(e)}")
        
        logger.info(f"Processing complete. Successful: {processed_files}, Failed: {failed_files}")

In [7]:
processor = CarDataProcessor(db_path="./car_specs_db")


2025-02-11 19:55:17,260 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [8]:
#processor.process_single_file("files/2025_buick_encore.csv")


In [9]:
processor.process_directory("files/")

2025-02-11 19:55:17,529 - INFO - Processing 2025 buick encore
2025-02-11 19:55:29,435 - INFO - Successfully processed 2025 buick encore
2025-02-11 19:55:29,436 - INFO - Processing 2025 honda hr-v
2025-02-11 19:55:42,432 - INFO - Successfully processed 2025 honda hr-v
2025-02-11 19:55:42,433 - INFO - Processing 2025 kia sportage
2025-02-11 19:55:54,005 - INFO - Successfully processed 2025 kia sportage
2025-02-11 19:55:54,005 - INFO - Processing complete. Successful: 3, Failed: 0


## Comparison Agent

In [10]:
class CarComparisonTool:
    def __init__(self, db_path: str = "./chromadb"):
        """Initialize the CarComparisonTool with ChromaDB connection."""
        self.client = chromadb.PersistentClient(path=db_path)
        
        # Create collections for each section if they don't exist
        self.specs_collection = self.client.get_or_create_collection(
            name="car_specifications",
            metadata={"description": "Basic car specifications"}
        )
        self.features_collection = self.client.get_or_create_collection(
            name="car_features",
            metadata={"description": "Car features and availability"}
        )
        self.safety_collection = self.client.get_or_create_collection(
            name="car_safety",
            metadata={"description": "Car safety ratings"}
        )

    # [Previous methods for file processing remain the same]
    
    def get_car_specs(self, year: str, make: str, model: str) -> Dict[str, str]:
        """Retrieve specifications for a specific car."""
        results = self.specs_collection.get(
            where={
                "$and": [
                    {"year": {"$eq": year}},
                    {"make": {"$eq": make}},
                    {"model": {"$eq": model}}
                ]
            }
        )
        
        specs_dict = {}
        for metadata in results['metadatas']:
            specs_dict[metadata['spec']] = metadata['value']
        return specs_dict

    def get_car_features(self, year: str, make: str, model: str) -> Dict[str, Dict[str, str]]:
        """Retrieve features and their availability for a specific car."""
        results = self.features_collection.get(
            where={
                "$and": [
                    {"year": {"$eq": year}},
                    {"make": {"$eq": make}},
                    {"model": {"$eq": model}}
                ]
            }
        )
        
        features_dict = {}
        for metadata in results['metadatas']:
            features_dict[metadata['feature']] = {
                'availability': metadata['availability'],
                'msrp': metadata['msrp']
            }
        return features_dict

    def get_car_safety(self, year: str, make: str, model: str) -> Dict[str, str]:
        """Retrieve safety ratings for a specific car."""
        results = self.safety_collection.get(
            where={
                "$and": [
                    {"year": {"$eq": year}},
                    {"make": {"$eq": make}},
                    {"model": {"$eq": model}}
                ]
            }
        )
        
        safety_dict = {}
        for metadata in results['metadatas']:
            safety_dict[metadata['safety_aspect']] = metadata['rating']
        return safety_dict

    def compare_specs(self, cars: List[Tuple[str, str, str]], specs_list: Optional[List[str]] = None) -> str:
        """
        Compare specifications between multiple cars.
        
        Args:
            cars: List of (year, make, model) tuples
            specs_list: Optional list of specific specs to compare. If None, compares all specs.
        
        Returns:
            Formatted comparison table as string
        """
        comparison_data = []
        headers = ['Specification']
        
        # Add car names to headers
        for year, make, model in cars:
            headers.append(f"{year} {make} {model}")
        
        # Get specs for all cars
        all_specs = {}
        for year, make, model in cars:
            specs = self.get_car_specs(year, make, model)
            all_specs[(year, make, model)] = specs
        
        # If no specific specs requested, use all available specs
        if specs_list is None:
            specs_list = set()
            for specs in all_specs.values():
                specs_list.update(specs.keys())
            specs_list = sorted(list(specs_list))
        
        # Build comparison rows
        for spec in specs_list:
            row = [spec]
            for car in cars:
                value = all_specs[car].get(spec, 'N/A')
                row.append(value)
            comparison_data.append(row)
        
        return tabulate(comparison_data, headers=headers, tablefmt='grid')

    def compare_features(self, cars: List[Tuple[str, str, str]], feature_list: Optional[List[str]] = None) -> str:
        """
        Compare features between multiple cars.
        
        Args:
            cars: List of (year, make, model) tuples
            feature_list: Optional list of specific features to compare. If None, compares all features.
        
        Returns:
            Formatted comparison table as string
        """
        comparison_data = []
        headers = ['Feature']
        
        # Add car names to headers
        for year, make, model in cars:
            headers.append(f"{year} {make} {model}")
        
        # Get features for all cars
        all_features = {}
        for year, make, model in cars:
            features = self.get_car_features(year, make, model)
            all_features[(year, make, model)] = features
        
        # If no specific features requested, use all available features
        if feature_list is None:
            feature_list = set()
            for features in all_features.values():
                feature_list.update(features.keys())
            feature_list = sorted(list(feature_list))
        
        # Build comparison rows
        for feature in feature_list:
            row = [feature]
            for car in cars:
                feature_info = all_features[car].get(feature, {})
                availability = feature_info.get('availability', 'N/A')
                msrp = feature_info.get('msrp', 'N/A')
                if msrp != '0':
                    value = f"{availability} (${msrp})"
                else:
                    value = availability
                row.append(value)
            comparison_data.append(row)
        
        return tabulate(comparison_data, headers=headers, tablefmt='grid')

    def compare_safety(self, cars: List[Tuple[str, str, str]]) -> str:
        """
        Compare safety ratings between multiple cars.
        
        Args:
            cars: List of (year, make, model) tuples
        
        Returns:
            Formatted comparison table as string
        """
        comparison_data = []
        headers = ['Safety Feature']
        
        # Add car names to headers
        for year, make, model in cars:
            headers.append(f"{year} {make} {model}")
        
        # Get safety ratings for all cars
        all_ratings = {}
        safety_aspects = set()
        for year, make, model in cars:
            ratings = self.get_car_safety(year, make, model)
            all_ratings[(year, make, model)] = ratings
            safety_aspects.update(ratings.keys())
        
        # Build comparison rows
        for aspect in sorted(safety_aspects):
            row = [aspect]
            for car in cars:
                rating = all_ratings[car].get(aspect, 'N/A')
                row.append(rating)
            comparison_data.append(row)
        
        return tabulate(comparison_data, headers=headers, tablefmt='grid')

    def compare_cars(self, cars: List[Tuple[str, str, str]], 
                    compare_specs: bool = True,
                    compare_features: bool = True,
                    compare_safety: bool = True,
                    specs_list: Optional[List[str]] = None,
                    feature_list: Optional[List[str]] = None) -> str:
        """
        Generate a comprehensive comparison between multiple cars.
        
        Args:
            cars: List of (year, make, model) tuples
            compare_specs: Whether to include specifications comparison
            compare_features: Whether to include features comparison
            compare_safety: Whether to include safety ratings comparison
            specs_list: Optional list of specific specs to compare
            feature_list: Optional list of specific features to compare
        
        Returns:
            Formatted comparison report as string
        """
        report = []
        
        if compare_specs:
            report.append("=== Specifications Comparison ===")
            report.append(self.compare_specs(cars, specs_list))
            report.append("\n")
        
        if compare_features:
            report.append("=== Features Comparison ===")
            report.append(self.compare_features(cars, feature_list))
            report.append("\n")
        
        if compare_safety:
            report.append("=== Safety Ratings Comparison ===")
            report.append(self.compare_safety(cars))
        
        return "\n".join(report)

In [11]:
comparison_tool = CarComparisonTool(db_path="./car_specs_db")


In [12]:
# Compare two cars
cars_to_compare = [
    ('2025', 'honda', 'hrv'),
    ('2025', 'buick', 'encore')
]

# Compare specific specs
specs_comparison = comparison_tool.compare_specs(
    cars_to_compare,
    specs_list=['Length', 'Width', 'Height', 'Engine Name']
)
print(specs_comparison)

+-----------------+------------------+-------------------------+
| Specification   | 2025 honda hrv   | 2025 buick encore       |
+=================+==================+=========================+
| Length          | N/A              | 171.2                   |
+-----------------+------------------+-------------------------+
| Width           | N/A              | 71.4                    |
+-----------------+------------------+-------------------------+
| Height          | N/A              | 64.1                    |
+-----------------+------------------+-------------------------+
| Engine Name     | N/A              | "1.2L inline 3-cylinder |
+-----------------+------------------+-------------------------+


In [13]:
full_comparison = comparison_tool.compare_cars(
    cars_to_compare
)
print(full_comparison)

=== Specifications Comparison ===
+------------------------+------------------+------------------------------------+
| Specification          | 2025 honda hrv   | 2025 buick encore                  |
+========================+==================+====================================+
| Body Style             | N/A              | SUV                                |
+------------------------+------------------+------------------------------------+
| Cargo Capacity         | N/A              | 23.5/50.2                          |
+------------------------+------------------+------------------------------------+
| Class                  | N/A              | N/A                                |
+------------------------+------------------+------------------------------------+
| Curb Weight            | N/A              | 3023                               |
+------------------------+------------------+------------------------------------+
| Cylinder Configuration | N/A              | in-line

## Car Comparison LLM

In [15]:
# self.client = anthropic.Anthropic(api_key=api_key)

In [55]:
class CarComparisonLLMAgent:
    def __init__(self, api_key: str, db_path: str = "./car_specs_db"):
        """Initialize the AI agent with Claude integration and comparison tool."""
        self.client = anthropic.Anthropic(api_key=api_key)
        self.comparison_tool = CarComparisonTool(db_path=db_path)
        self.debugging = False
        
        # Define feature categories and related terms
        self.feature_categories = {
            'tech_features': [
                'navigation', 'bluetooth', 'internet', 'wifi', 'smartphone', 
                'connectivity', 'audio', 'infotainment', 'display', 'screen',
                'usb', 'charging', 'mobile', 'android auto', 'apple carplay',
                'satellite radio', 'wireless', 'voice control', 'remote',
                'keyless', 'digital', 'smart'
            ],
            'safety_features': [
                'airbag', 'brake', 'collision', 'warning', 'assist', 'alert',
                'monitor', 'detection', 'emergency', 'safety', 'protection',
                'cruise control', 'lane', 'blind spot', 'parking', 'camera'
            ],
            'comfort_features': [
                'climate', 'air conditioning', 'heated', 'cooling', 'ventilated',
                'leather', 'seat', 'sunroof', 'moonroof', 'lumbar', 'automatic',
                'power', 'electric', 'adjustable'
            ]
        }
        
        # System prompt for query understanding
        self.system_prompt = """You are a car comparison expert assistant. Your task is to:
1. Extract car information (year, make, model) from user queries
2. Identify comparison criteria and feature categories they're interested in
3. Structure this information in a consistent JSON format

Available feature categories and examples:
- Tech features: navigation, bluetooth, connectivity, infotainment, displays, charging ports
- Safety features: airbags, collision warning, driver assistance, parking sensors
- Comfort features: climate control, heated seats, sunroof, power adjustments

Your response MUST be a valid JSON object with this EXACT structure:
{
    "cars": [
        {"year": "YYYY", "make": "Make", "model": "Model"}
    ],
    "criteria": {
        "specs": [],
        "features": [],
        "feature_categories": ["tech_features", "safety_features", "comfort_features"],
        "safety": false
    }
}

When the query mentions:
- Tech or technology: include "tech_features" in feature_categories
- Safety features: include "safety_features" in feature_categories
- Comfort or convenience: include "comfort_features" in feature_categories
- Space/room: include relevant dimension specs
"""

    def normalize_car_info(self, year: str, make: str, model: str) -> Tuple[str, str, str]:
        """Normalize car information to match database storage format."""
        make = make.lower()  # Store makes in lowercase
        model = model.lower()  # Store models in lowercase
        return (year, make, model)

    def parse_query_with_llm(self, query: str) -> Dict:
        """Use Claude to parse the natural language query."""
        try:
            message = self.client.messages.create(
                model="claude-3-sonnet-20240229",
                max_tokens=1024,
                temperature=0,
                system=self.system_prompt,
                messages=[
                    {
                        "role": "user",
                        "content": f"""Parse this car comparison query: {query}
                        
                        Note: If the query mentions 'space' or 'room', include these specific specs:
                        - Length
                        - Width
                        - Height
                        - Cargo Capacity
                        - Front Headroom
                        - Rear Headroom
                        - Front Legroom
                        - Rear Legroom"""
                    }
                ]
            )
            
            # Extract JSON from Claude's response
            response = message.content[0].text
            # Find JSON block in the response
            json_start = response.find('{')
            json_end = response.rfind('}') + 1
            if json_start >= 0 and json_end > json_start:
                json_str = response[json_start:json_end]
                result = json.loads(json_str)
                
                # If space is mentioned, ensure we include all relevant specs
                if 'space' in query.lower():
                    space_specs = ['Length', 'Width', 'Height', 'Cargo Capacity',
                                 'Front Headroom', 'Rear Headroom', 
                                 'Front Legroom', 'Rear Legroom']
                    if 'specs' not in result['criteria']:
                        result['criteria']['specs'] = []
                    result['criteria']['specs'].extend(space_specs)
                    # Remove duplicates while preserving order
                    result['criteria']['specs'] = list(dict.fromkeys(result['criteria']['specs']))
                
                return result
            else:
                raise ValueError("No valid JSON found in LLM response")
            
        except Exception as e:
            raise Exception(f"Error parsing query with LLM: {str(e)}")

    def generate_natural_response(self, comparison_data: str, query_context: Dict) -> str:
        """Use Claude to generate a natural language response from comparison data."""
        prompt = f"""You are a car expert assistant. Review the following comparison data and provide a 
        detailed analysis focusing on the aspects the user asked about.

        Raw comparison data to analyze:
        ```
        {comparison_data}
        ```

        Original query aspects of interest:
        {json.dumps(query_context.get('criteria', {}), indent=2)}

        Instructions:
        1. Look at the actual comparison data above carefully - it contains real specifications and ratings
        2. Provide specific numbers and details from the data
        3. Compare the values directly when available
        4. If a specific value is "N/A", mention that it's not available
        5. For safety ratings, mention the specific NHTSA ratings when shown
        6. For dimensions, include the actual measurements
        7. DO NOT make general statements about the manufacturers or models - stick to the data shown

        Format your response as a natural conversation, but ensure every comparison you make is based on 
        the actual numbers and ratings in the data above."""

        try:
            message = self.client.messages.create(
                model="claude-3-sonnet-20240229",
                max_tokens=1024,
                temperature=0.7,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )
            
            return message.content[0].text
            
        except Exception as e:
            return f"Error generating natural response: {str(e)}"

    def get_features_by_category(self, features_dict: Dict[str, Dict[str, str]], categories: List[str]) -> Dict[str, List[Dict[str, str]]]:
        """Categorize features based on their types."""
        categorized_features = {category: [] for category in categories}
        
        for feature_name, feature_info in features_dict.items():
            feature_lower = feature_name.lower()
            
            # Check each category's keywords
            for category, keywords in self.feature_categories.items():
                if category in categories:  # Only check requested categories
                    if any(keyword in feature_lower for keyword in keywords):
                        categorized_features[category].append({
                            'name': feature_name,
                            'availability': feature_info['availability'],
                            'msrp': feature_info['msrp']
                        })
        
        return categorized_features

    def format_features_comparison(self, cars_to_compare: List[Tuple[str, str, str]], 
                                 feature_categories: List[str]) -> str:
        """Format features comparison by category."""
        report_lines = []
        
        for year, make, model in cars_to_compare:
            features = self.comparison_tool.get_car_features(year, make, model)
            categorized = self.get_features_by_category(features, feature_categories)
            
            for category in feature_categories:
                if categorized[category]:
                    report_lines.append(f"\n=== {category.replace('_', ' ').title()} ===")
                    report_lines.append(f"For {year} {make.title()} {model.upper()}:")
                    for feature in categorized[category]:
                        report_lines.append(f"- {feature['name']}: {feature['availability']}")
                        if feature['msrp'] != '0':
                            report_lines.append(f"  MSRP: ${feature['msrp']}")
        
        return "\n".join(report_lines)

    def process_query(self, query: str) -> str:
        """Process a natural language query with LLM assistance."""
        try:
            # Parse query using LLM
            parsed_query = self.parse_query_with_llm(query)
            
            # Extract cars to compare and normalize their info
            cars_to_compare = [
                self.normalize_car_info(car['year'], car['make'], car['model'])
                for car in parsed_query['cars']
            ]
            
            if len(cars_to_compare) < 2:
                return "Please specify at least two cars to compare."
            
            # Debug logging for normalized car info
            if self.debugging:
                print("\nNormalized car info for comparison:")
                for year, make, model in cars_to_compare:
                    print(f"Year: {year}, Make: {make}, Model: {model}")
            
            # Get comparison data based on criteria with default values
            criteria = parsed_query.get('criteria', {})
            if not isinstance(criteria, dict):
                criteria = {}
            
            specs = criteria.get('specs', [])
            features = criteria.get('features', [])
            feature_categories = criteria.get('feature_categories', [])
            safety = criteria.get('safety', False)
            
            # Debug logging
            if self.debugging:
                print(f"\nParsed criteria: {criteria}")
                print(f"Feature categories: {feature_categories}")
            
            # Generate focused comparison
            report = []
            
            if specs:
                report.append("=== Specifications Comparison ===")
                specs_comparison = self.comparison_tool.compare_specs(
                    cars_to_compare,
                    specs_list=specs
                )
                report.append(specs_comparison)
                report.append("\n")
            
            if feature_categories:
                report.append("=== Features Comparison by Category ===")
                features_comparison = self.format_features_comparison(
                    cars_to_compare,
                    feature_categories
                )
                report.append(features_comparison)
                report.append("\n")
            
            if safety:
                report.append("=== Safety Ratings Comparison ===")
                safety_comparison = self.comparison_tool.compare_safety(cars_to_compare)
                report.append(safety_comparison)
                report.append("\n")
            
            comparison_data = "\n".join(report)
            
            # Generate natural language response
            return self.generate_natural_response(comparison_data, parsed_query)
            
        except Exception as e:
            return f"Sorry, I encountered an error: {str(e)}"

In [51]:
agent = CarComparisonLLMAgent(os.getenv("ANTHROPIC_API_KEY"), db_path="./car_specs_db")

In [52]:
queries = [
    "How do the 2025 Honda HR-V and 2025 Buick Encore compare in terms of space and safety?",
    "Which has better fuel economy and more features, the 2025 Kia Sportage or 2025 Honda HR-V?",
    "Compare the safety ratings and tech features of 2025 Buick Encore vs 2025 Honda HR-V"
]

In [56]:
queries = [
    "Which has better fuel economy and more features, the 2025 Kia Sportage or 2025 Honda HR-V?"
]

In [57]:
for query in queries:
    result = agent.process_query(query)
    
    print(query)
    print(result)


Parsed Query Result: {
  "cars": [
    {
      "year": "2025",
      "make": "Kia",
      "model": "Sportage"
    },
    {
      "year": "2025",
      "make": "Honda",
      "model": "HR-V"
    }
  ],
  "criteria": {
    "specs": [],
    "features": [],
    "feature_categories": [
      "tech_features",
      "safety_features",
      "comfort_features"
    ],
    "safety": false,
    "spec_categories": [
      "fuel_economy"
    ]
  }
}

Available specs in data:
[
  "Length",
  "Width",
  "Height",
  "Wheelbase",
  "Seating",
  "Cargo Capacity",
  "Front Headroom",
  "Rear Headroom",
  "Front Legroom",
  "Rear Legroom",
  "Front Shoulder Room",
  "Rear Shoulder Room",
  "Curb Weight",
  "Engine Name",
  "Trim",
  "Horsepower",
  "Body Style",
  "Fuel",
  "Transmission",
  "Class",
  "Standard MPG",
  "Drivetrain",
  "Cylinder Configuration",
  "Driving Range",
  "Engine Size",
  "Ground Clearance",
  "GVWR",
  "Number Of Cylinders",
  "Payload Capacity",
  "Std Mpg With Units",
  "Tor